# 06 - Fine-tune the RoBERTa-LoRA classifier

This notebook reproduces the classifier used by the RAG pipeline:

* `roberta-base` with a rank-8 LoRA adapter (`alpha=16`, dropout `0.1`).
* Weighted cross-entropy to handle class imbalance.
* 10 epochs, learning rate `1e-4`, batch size 16, warmup ratio `0.06`.
* Model selection uses validation precision; a 15% stratified validation split
  is carved out of the training corpus.

The resulting adapter is written to `models/roberta_lora_adapter_retrained/`,
which is excluded from Git. The released rank-8 adapter ships in
`models/roberta_lora_adapter/`.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix

from spam_detection.classifier import (
    create_lora_classifier,
    fine_tune_adapter,
    predict_roberta_proba,
    split_train_validation,
)
from spam_detection.config import (
    CLASSIFIER_TEST_PATH,
    CLASSIFIER_TRAIN_PATH,
    MODELS_DIR,
)
from spam_detection.evaluation import binary_metrics
from spam_detection.utils import set_seed

set_seed(42)

RETRAINED_ADAPTER_PATH = MODELS_DIR / "roberta_lora_adapter_retrained"

## Load the prepared corpus

In [ ]:
train = pd.read_csv(CLASSIFIER_TRAIN_PATH)
test = pd.read_csv(CLASSIFIER_TEST_PATH)
train_df, val_df = split_train_validation(train, test_size=0.15, seed=42)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test.shape)

## Create a fresh LoRA adapter

In [ ]:
model, tokenizer = create_lora_classifier(rank=8, alpha=16, dropout=0.1, device="cuda")
model.print_trainable_parameters()

## Train

`fine_tune_adapter` prepares tokenised datasets, computes balanced class
weights and runs the custom weighted-loss Trainer.

In [ ]:
trainer, model, tokenizer = fine_tune_adapter(
    train_df,
    val_df,
    model=model,
    tokenizer=tokenizer,
    output_dir=str(RETRAINED_ADAPTER_PATH.parent / "training_run"),
    epochs=10,
    learning_rate=1e-4,
    batch_size=16,
    max_length=256,
    warmup_ratio=0.06,
    metric_for_best_model="precision",
    seed=42,
)

## Save the adapter

In [ ]:
import torch

model.eval()
RETRAINED_ADAPTER_PATH.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(RETRAINED_ADAPTER_PATH))
tokenizer.save_pretrained(str(RETRAINED_ADAPTER_PATH))
print("Saved adapter to", RETRAINED_ADAPTER_PATH)

## Evaluate on the test split

In [ ]:
probabilities = predict_roberta_proba(test["text"].tolist(), model, tokenizer, batch_size=16)
predictions = (probabilities >= 0.5).astype(int)

metrics = binary_metrics(test["label"], predictions, probabilities)
print({key: round(value, 4) for key, value in metrics.items()})
print(classification_report(test["label"], predictions, target_names=["real", "fake"], digits=4))
print(confusion_matrix(test["label"], predictions))